## Imports and Constants

In [ ]:
import pynq
import numpy as np
import json
import os

# Import the driver and utilities from the local pynq_driver.py file
# This assumes pynq_driver.py is in the same directory as this notebook
from pynq_driver import DeepSoCFlowPYNQ

# --- Define Paths ---
# Assumes the notebook is in a directory that contains cgra4ml.bit, pynq_driver.py, y_exp.txt, and the 'vectors' folder.
NOTEBOOK_DIR = os.getcwd() 
BITSTREAM_PATH = os.path.join(NOTEBOOK_DIR, 'cgra4ml.bit')
VECTORS_DIR = os.path.join(NOTEBOOK_DIR, 'vectors')
CONFIG_PATH = os.path.join(VECTORS_DIR, 'config.json')
WBX_PATH = os.path.join(VECTORS_DIR, 'wbx.bin')
Y_EXP_PATH = os.path.join(NOTEBOOK_DIR, 'y_exp.txt') 

# --- Check that files exist before proceeding ---
assert os.path.exists(BITSTREAM_PATH), f"Bitstream not found at {BITSTREAM_PATH}"
assert os.path.exists(CONFIG_PATH), f"Config file not found at {CONFIG_PATH}"
assert os.path.exists(WBX_PATH), f"WBX file not found at {WBX_PATH}"
assert os.path.exists(Y_EXP_PATH), f"Expected output file not found at {Y_EXP_PATH}"

print("Setup complete. All necessary files found.")


## Load the Overlay

In [ ]:
print("Loading overlay...")
overlay = pynq.Overlay(BITSTREAM_PATH)
print("Overlay loaded successfully.")

#overlay.ip_dict

## Load Data and Allocate Buffers

In [ ]:
print("--- Testing Driver Initialization and Memory Allocation ---")

ACCELERATOR_IP_NAME = 'axi_cgra4ml_0'

driver = DeepSoCFlowPYNQ(overlay, config_path=CONFIG_PATH, accelerator_ip_name=ACCELERATOR_IP_NAME)

print("\n--- Driver initialization and memory allocation test complete! ---")

## Setup the model 

In [ ]:
print("\n--- Testing Model Setup ---")
driver.model_setup(wbx_path=WBX_PATH)
print("\n--- Model setup test complete! ---")


--- Testing Model Setup ---

Setting up model from vectors/wbx.bin...
Data copy complete.
Pre-loading all bundle parameters...
Parameter loading complete.
Register configuration complete.
Model setup finished.

--- Model setup test complete! ---


## Model Run

In [ ]:
print("\n--- Testing Model Run ---")

# Run the model. This will use the input data loaded from wbx.bin
output = driver.model_run()

print("\n--- Model run test complete! ---")
print(f"Output shape: {output.shape}")
print(f"Output dtype: {output.dtype}")
print(f"First 10 output values:\n{output[:10]}")

print("\n--- Comparing PYNQ Output with Expected Output ---")
# We already know it's float32 from your y_exp.txt, but for robustness:
last_bundle = driver.bundles[-1]
output_dtype = np.float32 if last_bundle['is_softmax'] else np.int32

expected_output = np.loadtxt(Y_EXP_PATH, dtype=output_dtype)

print(f"Expected output shape: {expected_output.shape}")
print(f"Expected output dtype: {expected_output.dtype}")
print(f"First 10 expected values:\n{expected_output[:10]}")

# Compare the actual output from the PYNQ driver with the expected output
is_close = np.allclose(output, expected_output, atol=1e-2, rtol=1e-2) # Tolerances may need tuning
print(f"\nOutputs are numerically close: {is_close}")

if not is_close:
    print("\nDifferences (first 20 values where they differ or beyond tolerance):")
    # Find indices where values are NOT close
    diff_idx = np.where(~np.isclose(output, expected_output, atol=1e-2, rtol=1e-2))[0]
    if diff_idx.size > 0:
        # This check prevents the IndexError if output is smaller than expected
        max_len = max(len(output), len(expected_output))
        for i in range(max_len):
            if i >= 20: break # Limit to first 20 diffs
            pynq_val = output[i] if i < len(output) else "N/A (out of bounds)"
            exp_val = expected_output[i] if i < len(expected_output) else "N/A (out of bounds)"
            
            if not np.isclose(pynq_val if isinstance(pynq_val, (int, float)) else 0, 
                              exp_val if isinstance(exp_val, (int, float)) else 0, 
                              atol=1e-2, rtol=1e-2):
                diff = abs(pynq_val - exp_val) if isinstance(pynq_val, (int, float)) and isinstance(exp_val, (int, float)) else "N/A"
                print(f"  Index {i}: PYNQ={pynq_val}, Expected={exp_val}, Diff={diff}")
    else:
        print("No significant differences found beyond tolerance, but np.allclose returned False.")

print("\n--- Comparison Complete ---")



## Cleanup

In [ ]:
print("\nCleaning up resources...")
# The try/except blocks handle cases where the notebook is partially run
try:
    if 'driver' in locals() and driver is not None:
        del driver
except NameError:
    pass

try:
    if 'overlay' in locals() and overlay is not None:
        overlay.free()
except NameError:
    pass

print("Cleanup complete.")



Cleaning up resources...

Releasing memory buffers.
Cleanup complete.
